In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Hydra Configuration
import hydra
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from omegaconf import OmegaConf, DictConfig

from pathlib import Path

# Clear any existing Hydra instance
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()

# Get the absolute path to the configs directory
import os
repo_root = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
config_dir = (repo_root / "configs").resolve()

# Initialize Hydra with the config directory
with initialize_config_dir(version_base=None, config_dir=str(config_dir)):
    # Compose configuration
    cfg = compose(
        config_name="config",
        overrides=[
            # Example overrides (can be modified by user)
            # "dataset=cunit",
            # "stage2=disk_superglue",
            "reset=mvs_only",
        ]
    )

# Print configuration for verification
print("="*80)
print("Loaded Configuration:")
print("="*80)
print(OmegaConf.to_yaml(cfg))
print("="*80)

# Keep reference to config
Config = cfg

output_root = Path(Config.output_root)
raw_images_path = Path(Config.dataset.raw_images)

paths = {
    "resized_images": raw_images_path / "resized",
    "features": output_root / "features.h5",
    "matches": output_root / "matches.h5",
    "pairs": output_root / "pairs.txt",
    "fine_sfm": output_root / f"sfm_{Config.stage2.extractor}+{Config.stage2.matcher}",
    # New paths for visualization
    "pairs_loc": output_root / "pairs_loc.txt"
}

# Visualization

After you have created the outputs using the pipeline you might use this notebook to qualitatively insepct the output

In [ ]:
from hloc import colmap_from_nvm, triangulation, localize_sfm, visualization, extract_features, match_features, pairs_from_exhaustive

from hloc.utils import viz_3d

import pycolmap

Load the model

In [ ]:
# Make sure to select the biggest model reconstructed
model_path = paths["fine_sfm"] / "0"

model = pycolmap.Reconstruction(model_path)

images = paths["resized_images"]

Color the keypoints by visibility: blue if sucessfully triangulated, red if never matched.

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="visibility", n=3)

Color the keypoints by track length: red keypoints are observed many times, blue keypoints few.

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="track_length", n=3)

Depth

In [ ]:
visualization.visualize_sfm_2d(model, images, color_by="depth", n=3)

## Localization

The following might be usefull to when thinking about extending on existing models, but with a new camera or extremly different lightning conditions, or when a scene might have changed drastically (construction)

- Use EXIF data to guess camera parameters `infer_camera_from_image`
- Use Perspective-n-Point algorithm together with RANSAC to loclize image `QueryLocalizer` 

In [ ]:
# List all images used to create 3D model 

ext_filter = [".jpg", ".jpeg", ".png"]

references = [f.name for f in images.iterdir() if f.is_file() and f.suffix.lower() in ext_filter]

In [ ]:
# Get a query image 

query = "query/query_image.JPG"

feature_conf = extract_features.confs[Config.stage2.extractor]
feature_conf["preprocessing"]["resize_force"] = False  # We already resized the images, no need to force resize again

# Extract and Match Query features

# 2. Extract SuperPoint Features

feature_path = extract_features.main(
    feature_conf, 
    images, 
    image_list =[query],
    feature_path= paths["features"] ,
    overwrite=True
)

# 3. Generate Pairs from Coarse Poses
pairs_from_exhaustive.main(paths["pairs_loc"], image_list=[query], ref_list=references)

# 4. Match Features (LightGlue)
matcher_conf = match_features.confs["superpoint+lightglue"]
match_path = match_features.main(
    matcher_conf, 
    paths["pairs_loc"], 
    features=paths["features"], 
    matches=paths["matches"],
    overwrite=True
)

In [ ]:
from hloc.localize_sfm import QueryLocalizer, pose_from_cluster

camera = pycolmap.infer_camera_from_image(images / query)
ref_ids = [model.find_image_with_name(r).image_id for r in references]
conf = {
    "estimation": {"ransac": {"max_error": 50, "min_num_trials": 1000}},
    "refinement": {"refine_focal_length": True, "refine_extra_params": True},
}
localizer = QueryLocalizer(model, conf)
ret, log = pose_from_cluster(localizer, query, camera, ref_ids, paths["features"], paths["matches"])

print(f'found {ret["num_inliers"]}/{len(ret["inlier_mask"])} inlier correspondences.')
visualization.visualize_loc_from_log(images, query, log, model, top_k_db=3, target_db_name="HYPERLAPSE_0001.JPG")

NOTE: Unclear why so many outliers at the moment

Visualize 3D

In [ ]:
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(
    fig, model, color="rgba(255,0,0,0.5)", name="mapping", points_rgb=True
)
fig.show()

In [ ]:
import numpy as np

viz_3d.plot_camera_colmap(
    fig, ret["cam_from_world"], ret["camera"], color="rgba(0,255,0,0.5)", name=query, fill=True,
    text=f"inliers: {ret['num_inliers']} / {ret['inlier_mask'].shape[0]}"
)
# visualize 2D-3D correspodences
inl_3d = np.array(
    [model.points3D[pid].xyz for pid in np.array(log["points3D_ids"])[ret["inlier_mask"]]]
)
viz_3d.plot_points(fig, inl_3d, color="lime", ps=1, name=query)
fig.show()